<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">5. Lakehouse Federation</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 5.4 Demo Creating a Foreign Catalog Connection to AWS Glue Catalog

This demo walks through federating an AWS Glue Catalog into Unity Catalog as a foreign catalog. The flow involves nine steps spread across two systems: AWS-side IAM role provisioning (steps 1, 3, 4) and Databricks-side credential/connection setup (steps 2, 5, 6, 7, 8, 9). Two IAM roles are created - one for Glue metadata access, one for S3 data access - which separates control-plane and data-plane permissions.

## Learning Objectives

By the end of this demonstration, you will be able to:
- Provision the AWS IAM roles needed for UC to read Glue metadata and S3 data with least-privilege permissions
- Register the IAM roles with Unity Catalog as `SERVICE` and `STORAGE` credentials via the Databricks CLI
- Update IAM trust policies with the UC-issued external ID to complete the secure credential exchange
- Create a UC `CONNECTION` and `FOREIGN CATALOG` over a Glue database
- Query Glue-resident tables directly from Databricks

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Video Demonstration</strong>
            <p style="margin: 8px 0 0 0; color: #333;">In the standard classroom environment, this demo is delivered as a <strong>video walkthrough</strong> because it requires an AWS account and an existing Glue Catalog, neither of which are available in the lab environment. The notebook below contains the complete working demo. If you have the required infrastructure, substitute <code>&lt;aws_account_id&gt;</code> with your AWS account ID and run end-to-end.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Required Permissions</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This demo creates IAM roles, storage credentials, external locations, connections, and a foreign catalog. The person running it needs:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><b>Databricks:</b> <b>Metastore Admin</b> (to create storage credentials, external locations, connections, and foreign catalogs).</li>
                <li><b>AWS:</b> <b>IAM admin</b> permissions in the target account (to create IAM roles and trust policies for both Glue access and S3 storage access).</li>
            </ul>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Requires Existing AWS Glue Catalog</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The target AWS account must have an existing Glue Catalog with tables and underlying data in S3. AWS-side steps (marked with the AWS logo) run in the AWS Console or CLI; Databricks-side steps (marked with the Databricks logo) run via the Databricks CLI/API or as SQL in this notebook.</p>
        </div>
    </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-Common

## Step 1. (AWS) Create the Glue Access IAM Role

In the target AWS account (programmatically or via the Console), create an IAM role named `DatabricksGlueAccessRole`. The trust policy temporarily uses placeholder external ID `0000` - it will be updated in Step 3 with the value Unity Catalog returns from the credential creation in Step 2. The permissions follow least-privilege: read-only access to Glue catalog metadata operations.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Initial Trust Policy (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="json">
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": [
                    "arn:aws:iam::414351767826:role/unity-catalog-prod-UCMasterRole-14S5ZJVKOTYTL",
                    "arn:aws:iam::{aws_account_id}:role/DatabricksGlueAccessRole"
                ]
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "sts:ExternalId": "0000"
                }
            }
        }
    ]
}
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Permissions Policy (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="json">
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "glue:GetDatabase",
                "glue:GetDatabases",
                "glue:GetTable",
                "glue:GetTables",
                "glue:GetPartition",
                "glue:GetPartitions",
                "glue:BatchGetPartition",
                "glue:GetCatalogImportStatus"
            ],
            "Resource": "*"
        }
    ]
}
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 2. (Databricks) Create the Service Credential

Register the Glue access IAM role with Unity Catalog as a `SERVICE` credential. UC will return an `external_id` in the response - capture it for Step 3, where the IAM role's trust policy is updated to require it.

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Capture the external_id</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The CLI response includes an <code>external_id</code> field. Copy it now - Step 3 substitutes it into the IAM trust policy condition.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Create Service Credential (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
databricks credentials create-credential \
  --json '{
    "name": "databricks_glue_service_credential",
    "purpose": "SERVICE",
    "aws_iam_role": {
      "role_arn": "arn:aws:iam::{aws_account_id}:role/DatabricksGlueAccessRole"
    }
  }'
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'Databricks');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 3. (AWS) Update the Glue Role Trust Policy

Replace the placeholder `0000` external ID in `DatabricksGlueAccessRole`'s trust policy with the `external_id` returned in Step 2. This completes the secure credential-exchange handshake: only UC, presenting that exact external ID, can assume the role.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Updated Trust Policy (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="json">
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": [
                    "arn:aws:iam::{aws_account_id}:role/DatabricksGlueAccessRole",
                    "arn:aws:iam::414351767826:role/unity-catalog-prod-UCMasterRole-14S5ZJVKOTYTL"
                ]
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "sts:ExternalId": "{external_id_from_step_2}"
                }
            }
        }
    ]
}
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 4. (AWS) Create the S3 Storage IAM Role

Glue access alone reaches metadata but not the underlying parquet files. Create a second IAM role - `DatabricksGlueStorageAccessRole` - scoped to the S3 bucket the Glue tables live on. Separating the metadata role from the storage role keeps each scope minimal and lets you grant data plane access independently of catalog access. As with Step 1, the trust policy starts with placeholder external ID `0000` and is updated after Step 5.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Initial Trust Policy (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="json">
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": [
                    "arn:aws:iam::{aws_account_id}:role/DatabricksGlueStorageAccessRole",
                    "arn:aws:iam::414351767826:role/unity-catalog-prod-UCMasterRole-14S5ZJVKOTYTL"
                ]
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "sts:ExternalId": "0000"
                }
            }
        }
    ]
}
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> S3 Read-Only Permissions Policy (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="json">
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket",
                "s3:GetBucketLocation"
            ],
            "Resource": [
                "arn:aws:s3:::crawler-public-us-east-1",
                "arn:aws:s3:::crawler-public-us-east-1/*"
            ]
        }
    ]
}
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 5. (Databricks) Create the Storage Credential

Register the storage IAM role with Unity Catalog as a `STORAGE` credential. As in Step 2, the response returns an `external_id` - capture it and update `DatabricksGlueStorageAccessRole`'s trust policy back in AWS (same pattern as Step 3, applied to the storage role).

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Create Storage Credential (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
databricks credentials create-credential \
  --json '{
    "name": "databricks_glue_storage_credential",
    "purpose": "STORAGE",
    "aws_iam_role": {
      "role_arn": "arn:aws:iam::{aws_account_id}:role/DatabricksGlueStorageAccessRole"
    }
  }'
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'Databricks');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 6. (Databricks) Create the External Location

An external location maps a logical UC name to a physical S3 path, paired with a storage credential. The foreign catalog will reference this location via `authorized_paths` in Step 8.

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Why <code>--skip-validation</code></strong>
            <p style="margin: 8px 0 0 0; color: #333;">The storage role intentionally has read-only S3 permissions (no <code>PutObject</code> / <code>DeleteObject</code>). UC validates external locations by performing a write-then-delete probe; without write permission the validation fails. <code>--skip-validation</code> bypasses the probe - safe here because the read permissions have been verified by IAM design.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Create External Location (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
# --skip-validation is required because the storage role intentionally has no write/delete permissions
databricks external-locations create \
  databricks_glue_external_location \
  's3://crawler-public-us-east-1/' \
  databricks_glue_storage_credential \
  --skip-validation
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-json.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'json';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'json' ? 'JSON' : 'Databricks');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Step 7. (Databricks) Create the Connection

The connection forms the logical bridge between Unity Catalog and AWS Glue, encapsulating the AWS account, region, and the Glue service credential created in Step 2. Substitute `<aws_account_id>` with your AWS account ID before running.

In [0]:
CREATE CONNECTION aws_glue_connection
  TYPE GLUE
  OPTIONS (
    aws_account_id '<aws_account_id>',
    aws_region 'us-east-1',
    credential 'databricks_glue_service_credential'
  );

In [0]:
DESCRIBE CONNECTION aws_glue_connection;

## Step 8. (Databricks) Create the Foreign Catalog

The foreign catalog ties everything together - the connection (which authenticates via the service credential) and the `authorized_paths` (which UC checks against external locations to decide which storage credential to vend at query time).

In [0]:
CREATE FOREIGN CATALOG aws_glue_catalog
  USING CONNECTION aws_glue_connection
  OPTIONS (authorized_paths 's3://crawler-public-us-east-1');

In [0]:
DESCRIBE CATALOG EXTENDED aws_glue_catalog;

## Step 9. (Databricks) Query the Foreign Catalog

With the connection and foreign catalog in place, Glue tables are queryable as if they were native UC tables. The query below targets a Glue database created via CloudFormation (`cfn-database-flights-1`) - substitute the database / table names that exist in your Glue catalog.

In [0]:
SELECT *
FROM aws_glue_catalog.`cfn-database-flights-1`.`cfn-manual-table-flights-1`
LIMIT 25;

## Migration Path

Foreign catalogs are read-only by design. The natural progression for tables that should ultimately be governed by UC end-to-end is foreign -> external -> managed - each step relinquishing more of the source-side dependency.

<div style="display: flex; align-items: center; justify-content: center; gap: 0; padding: 24px 16px; max-width: 900px; margin: 16px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; font-size: 1em">

  <div style="background: #fff3e0; border: 2px solid #ff9800; border-radius: 6px; padding: 14px 20px; text-align: center; line-height: 1.4; min-width: 180px"><div style="font-weight: 600">Foreign Table</div><div style="color: #555; margin-top: 4px">Read-only</div></div>

  <div style="display: flex; align-items: center; align-self: center; padding: 0 16px"><div style="width: 40px; height: 2px; background: #555"></div><div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div></div>

  <div style="background: #e3f2fd; border: 2px solid #1976d2; border-radius: 6px; padding: 14px 20px; text-align: center; line-height: 1.4; min-width: 180px"><div style="font-weight: 600">External Table</div><div style="color: #555; margin-top: 4px"><code>SET EXTERNAL</code></div></div>

  <div style="display: flex; align-items: center; align-self: center; padding: 0 16px"><div style="width: 40px; height: 2px; background: #555"></div><div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div></div>

  <div style="background: #e8f5e9; border: 2px solid #4caf50; border-radius: 6px; padding: 14px 20px; text-align: center; line-height: 1.4; min-width: 180px"><div style="font-weight: 600">Managed Table</div><div style="color: #555; margin-top: 4px"><code>SET MANAGED</code></div></div>

</div>

## Key Takeaways

Federating Glue into Unity Catalog uses the same `CONNECTION` + `FOREIGN CATALOG` pattern as 5.2's SQL Server and Snowflake demos, plus an extra layer for the IAM trust dance and external locations needed to reach the underlying S3 data.

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What This Demo Showed</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong>Two IAM roles</strong> separate concerns: <code>DatabricksGlueAccessRole</code> for catalog metadata, <code>DatabricksGlueStorageAccessRole</code> for S3 data - each scoped to least-privilege</li>
                <li><strong>External-ID handshake</strong> closes the trust loop: UC issues an external ID on credential creation, the IAM trust policy is updated to require that ID, only UC can assume the role</li>
                <li><strong>Two UC credentials</strong> wrap the IAM roles: <code>SERVICE</code> for catalog access, <code>STORAGE</code> for S3 access</li>
                <li><strong>External locations</strong> map S3 paths to storage credentials so UC knows which credential to vend at query time</li>
                <li><code>CONNECTION</code> + <code>FOREIGN CATALOG</code> with <code>authorized_paths</code> stitches everything together - Glue tables become queryable through UC with no data movement</li>
                <li>Foreign tables are <strong>read-only</strong>; the migration path foreign -&gt; external -&gt; managed lifts ownership progressively into UC</li>
            </ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>